# Online Shoppers — Gelismis Siniflandirma Modelleri ve Karsilastirma

Bu notebook, Logistic Regression baseline'i uzerine Random Forest ve
Gradient Boosting modelleri eklemekte; uclisini ayni test seti uzerinde
karsilastirmaktadir.
`Segment` ozelligi (Segmentation notebook'undan) her iki gelismis modele
ek girdi olarak verilmistir.

**Akis:**
1. Veri on isleme (baseline ile ayni pipeline + Segment ozelligi)
2. Baseline — Logistic Regression
3. Random Forest
4. Gradient Boosting
5. Model karsilastirmasi (Confusion Matrix, ROC, PR, metrik tablosu)
6. Stratified 5-Fold Cross Validation
7. Segment bazinda model performansi

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")

COLORS = ["#1565c0", "#2e7d32", "#e65100"]

In [ ]:
df_raw = pd.read_csv("cleaned.csv", index_col=0)
print(f"Ham veri boyutu: {df_raw.shape}")

In [ ]:
# Segmentasyon pipeline'i ile Segment ozelligi uret
_SEG_FEATURES = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues", "SpecialDay",
]
df_raw["_VisitorType_enc"] = df_raw["VisitorType"].map(
    {"Returning_Visitor": 2, "New_Visitor": 1, "Other": 0}
)
df_raw["_Weekend_enc"] = df_raw["Weekend"].astype(int)
_month_map = {"Feb": 2, "Mar": 3, "May": 5, "June": 6, "Jul": 7,
              "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}
df_raw["_Month_enc"] = df_raw["Month"].map(_month_map)

_seg_cols = _SEG_FEATURES + ["_VisitorType_enc", "_Weekend_enc", "_Month_enc"]
_sc = StandardScaler()
_X_km = _sc.fit_transform(df_raw[_seg_cols])

km = KMeans(n_clusters=3, random_state=42, n_init=10)
df_raw["Segment"] = km.fit_predict(_X_km)

df_raw.drop(columns=["_VisitorType_enc", "_Weekend_enc", "_Month_enc"], inplace=True)
print("Segment dagilimi:")
print(df_raw["Segment"].value_counts().sort_index())

In [ ]:
df_model = df_raw.copy()

# SpecialDay ikili degiskene donusturulur
df_model["Is_SpecialDay"] = (df_model["SpecialDay"] > 0).astype(int)
df_model.drop(columns=["SpecialDay"], inplace=True)

# BounceRates cikarilerek multicollinearity azaltilir
df_model.drop(columns=["BounceRates"], inplace=True)

df_model["Revenue"] = df_model["Revenue"].astype(int)

cat_cols = ["Month", "OperatingSystems", "Browser",
            "Region", "TrafficType", "VisitorType", "Weekend"]
df_model = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

# Baseline icin Segment icermeyen versiyon
X_base = df_model.drop(columns=["Revenue", "Segment"])
# Gelismis modeller icin Segment dahil versiyon
X_seg  = df_model.drop(columns=["Revenue"])
y      = df_model["Revenue"]

X_base_tr, X_base_te, y_tr, y_te = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)
X_seg_tr, X_seg_te, _, _ = train_test_split(
    X_seg, y, test_size=0.2, random_state=42, stratify=y
)

sc_base = StandardScaler()
X_base_tr_sc = sc_base.fit_transform(X_base_tr)
X_base_te_sc = sc_base.transform(X_base_te)

sc_seg = StandardScaler()
X_seg_tr_sc = sc_seg.fit_transform(X_seg_tr)
X_seg_te_sc = sc_seg.transform(X_seg_te)

print(f"Egitim seti (Segment'siz): {X_base_tr_sc.shape}")
print(f"Egitim seti (Segment'li) : {X_seg_tr_sc.shape}")
print(f"Test seti                : {X_base_te_sc.shape}")
print(f"\nSinif dagilimi (test):\n{y_te.value_counts().to_dict()}")

## Baseline — Logistic Regression

In [ ]:
baseline = LogisticRegression(
    random_state=42, class_weight="balanced", max_iter=1000
)
baseline.fit(X_base_tr_sc, y_tr)

y_pred_bl = baseline.predict(X_base_te_sc)
y_prob_bl = baseline.predict_proba(X_base_te_sc)[:, 1]

print("--- Logistic Regression ---")
print(classification_report(y_te, y_pred_bl, target_names=["No Revenue", "Revenue"]))
print(f"ROC-AUC      : {roc_auc_score(y_te, y_prob_bl):.4f}")
print(f"Avg Precision: {average_precision_score(y_te, y_prob_bl):.4f}")

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_seg_tr_sc, y_tr)

y_pred_rf = rf.predict(X_seg_te_sc)
y_prob_rf = rf.predict_proba(X_seg_te_sc)[:, 1]

print("--- Random Forest ---")
print(classification_report(y_te, y_pred_rf, target_names=["No Revenue", "Revenue"]))
print(f"ROC-AUC      : {roc_auc_score(y_te, y_prob_rf):.4f}")
print(f"Avg Precision: {average_precision_score(y_te, y_prob_rf):.4f}")

In [ ]:
feat_names = list(X_seg.columns)
importances_rf = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)

importances_rf.head(20).plot(
    kind="barh", color="steelblue", edgecolor="white", figsize=(10, 6)
)
plt.gca().invert_yaxis()
plt.title("Random Forest — En Onemli 20 Ozellik")
plt.xlabel("Onem Skoru")
plt.tight_layout()
plt.show()

In [ ]:
sample_weights = compute_sample_weight("balanced", y_tr)

gbm = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    min_samples_leaf=10,
    random_state=42,
)
gbm.fit(X_seg_tr_sc, y_tr, sample_weight=sample_weights)

y_pred_gbm = gbm.predict(X_seg_te_sc)
y_prob_gbm = gbm.predict_proba(X_seg_te_sc)[:, 1]

print("--- Gradient Boosting ---")
print(classification_report(y_te, y_pred_gbm, target_names=["No Revenue", "Revenue"]))
print(f"ROC-AUC      : {roc_auc_score(y_te, y_prob_gbm):.4f}")
print(f"Avg Precision: {average_precision_score(y_te, y_prob_gbm):.4f}")

In [ ]:
importances_gbm = pd.Series(gbm.feature_importances_, index=feat_names).sort_values(ascending=False)

importances_gbm.head(20).plot(
    kind="barh", color="darkorange", edgecolor="white", figsize=(10, 6)
)
plt.gca().invert_yaxis()
plt.title("Gradient Boosting — En Onemli 20 Ozellik")
plt.xlabel("Onem Skoru")
plt.tight_layout()
plt.show()

## Model Karsilastirmasi

In [ ]:
MODELS = {
    "Logistic Regression": (y_pred_bl, y_prob_bl),
    "Random Forest":       (y_pred_rf, y_prob_rf),
    "Gradient Boosting":   (y_pred_gbm, y_prob_gbm),
}

# Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, (pred, _)), color in zip(axes, MODELS.items(), COLORS):
    cm = confusion_matrix(y_te, pred, normalize="true")
    sns.heatmap(
        cm, annot=True, fmt=".2f", cmap="Blues", ax=ax,
        cbar=False, linewidths=0.4,
        xticklabels=["No Rev", "Revenue"],
        yticklabels=["No Rev", "Revenue"],
    )
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Tahmin")
    ax.set_ylabel("Gercek")

plt.suptitle("Confusion Matrix Karsilastirmasi (Normalize)", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for (name, (_, prob)), color in zip(MODELS.items(), COLORS):
    fpr, tpr, _ = roc_curve(y_te, prob)
    auc_val = roc_auc_score(y_te, prob)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})", color=color, linewidth=2)

axes[0].plot([0, 1], [0, 1], "k--", linewidth=1)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend(fontsize=9)

for (name, (_, prob)), color in zip(MODELS.items(), COLORS):
    prec, rec, _ = precision_recall_curve(y_te, prob)
    ap = average_precision_score(y_te, prob)
    axes[1].plot(rec, prec, label=f"{name} (AP={ap:.3f})", color=color, linewidth=2)

axes[1].axhline(y_te.mean(), color="gray", linestyle="--", linewidth=1,
                label=f"Sinif orani ({y_te.mean():.2f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
rows = []
for name, (pred, prob) in MODELS.items():
    rows.append({
        "Model":           name,
        "Precision (Rev)": round(precision_score(y_te, pred), 4),
        "Recall (Rev)":    round(recall_score(y_te, pred), 4),
        "F1 (Rev)":        round(f1_score(y_te, pred), 4),
        "ROC-AUC":         round(roc_auc_score(y_te, prob), 4),
        "Avg Precision":   round(average_precision_score(y_te, prob), 4),
    })

summary = pd.DataFrame(rows).set_index("Model")
print(summary.to_string())
summary

In [ ]:
metrics = ["Precision (Rev)", "Recall (Rev)", "F1 (Rev)", "ROC-AUC", "Avg Precision"]
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 5))
for i, (model_name, row) in enumerate(summary.iterrows()):
    bars = ax.bar(x + i * width, row[metrics].values, width,
                  label=model_name, color=COLORS[i], edgecolor="white", linewidth=0.8)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f"{bar.get_height():.3f}",
                ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics, fontsize=10)
ax.set_ylabel("Skor")
ax.set_title("Model Performans Karsilastirmasi")
ax.legend(fontsize=9)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

## Stratified 5-Fold Cross Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    "Logistic Regression": (baseline, X_base_tr_sc),
    "Random Forest":       (rf,       X_seg_tr_sc),
    "Gradient Boosting":   (gbm,      X_seg_tr_sc),
}

cv_results = {}
for name, (model, X_tr) in cv_models.items():
    scores = cross_val_score(model, X_tr, y_tr, cv=cv,
                             scoring="roc_auc", n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:<25} AUC: {scores.mean():.4f}  (+/- {scores.std():.4f})")

In [ ]:
pd.DataFrame(cv_results).boxplot(
    patch_artist=True,
    boxprops=dict(facecolor="#bbdefb"),
    medianprops=dict(color="red", linewidth=2),
    whiskerprops=dict(linewidth=1.5),
    capprops=dict(linewidth=1.5),
    figsize=(9, 4),
)
plt.ylabel("ROC-AUC")
plt.title("5-Fold CV ROC-AUC Dagilimi")
plt.tight_layout()
plt.show()

##  Segment Bazinda Model Performansi

In [ ]:
best_name = summary["ROC-AUC"].idxmax()
best_pred, best_prob = MODELS[best_name]
print(f"En iyi model: {best_name}")

test_idx = X_seg_te.index
seg_test = df_raw.loc[test_idx, "Segment"]

seg_perf = []
for seg in sorted(seg_test.unique()):
    mask = (seg_test == seg).values
    f1  = f1_score(y_te[mask], best_pred[mask], zero_division=0)
    auc = (roc_auc_score(y_te[mask], best_prob[mask])
           if y_te[mask].nunique() > 1 else float("nan"))
    seg_perf.append({
        "Segment":      seg,
        "N":            int(mask.sum()),
        "Revenue Orani": round(y_te[mask].mean(), 4),
        "F1":           round(f1, 4),
        "AUC":          round(auc, 4),
    })

seg_df = pd.DataFrame(seg_perf).set_index("Segment")
print(seg_df.to_string())
seg_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

seg_df[["F1", "AUC"]].plot(
    kind="bar", ax=axes[0],
    color=["steelblue", "darkorange"],
    edgecolor="white", rot=0,
)
axes[0].set_title(f"Segment Bazinda F1 & AUC ({best_name})")
axes[0].set_xlabel("Segment")
axes[0].set_ylim(0, 1.05)

axes[1].bar(seg_df.index, seg_df["Revenue Orani"] * 100,
            color="#c62828", edgecolor="white")
axes[1].set_title("Segment Bazinda Gercek Revenue Orani (%)")
axes[1].set_xlabel("Segment")
axes[1].set_ylabel("%")
axes[1].set_xticks(seg_df.index)
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## Bulgular

**Baseline (Logistic Regression)**
ROC-AUC ~0.91, ancak precision dusuk (~0.51).
Sinif agirlandirmasi recall'i yuksek tutmaktadir.

**Random Forest ve Gradient Boosting**
Her iki model de baseline'i gecmektedir.
Segment ozelligi marjinal bir AUC artisi saglamaktadir.
Cross Validation sonuclari test setiyle tutarlidir; overfitting izlenmemistir.